# Hidden Subsequences
![](https://live.staticflickr.com/65535/54327633282_6bc45ba42a_o.png)

*Image generated using the DALL-E model.*

## Introduction
From ancient soothsayers interpreting the arrangement of stars to modern cryptographers tracing hidden messages, humanity has always sought meaning in seemingly chaotic data. Sometimes, key information hides in small sequences of symbols, and their value is revealed only after precise analysis.

In this task, you will play the role of a detective, searching for structural dependencies in a set of binary strings. You will be given a dataset containing example strings and their correctly computed values. Your goal is to develop a method for analyzing hidden patterns that allows you to determine the values of strings not present in the dataset as accurately as possible.

We say that string $T$ is a subsequence of $S$ and denote $T \subseteq S$ if
$$
T = S_{i_1}S_{i_2} \dots S_{i_k}
$$
where
$$
1 \leq i_1 < i_2 < \dots < i_k \leq n
$$
For $k$ and $n$ being the lengths of strings $T$ and $S$ respectively, and indices ($i_1 < i_2 < \dots < i_k$) being a strictly increasing sequence of natural numbers (not necessarily consecutive).

The solution for a given binary string $S \in \{0,1\}^{n}$ and a defined set containing a pattern and its weight $W = \{(T, v):T \in \{0,1\}^{k}, k \le n, v \in Z\}$ is the number
$$
\phi(S) = \sum_{(T_{i}, v_{i}) \in W} v_{i} \cdot \text{I} (T_{i})
$$
where $\text{I}(T_{i}) = \begin{cases}
1, & T_{i} \subseteq S \\
0, & T_{i} \subsetneq S
\end{cases}$

In other words, $\phi(S)$ is the sum of the values of all strings in the set $W$ that are subsequences of $S$.

**Example:**
For the set $W = \{(1111, 1), (1010, 2)\}$, we have:
- $\phi$(0**1111**000) = 1
- $\phi$(1**10**00**1**0**0**) = 2
- $\phi$(0**110110**0) = 3, because

  - 1111 $\subseteq$ 0**11**0**11**00

  - 1010 $\subseteq$ 01**101**1**0**0
- $\phi$(01100000) = 0, because 1111, 1010 $\subsetneq$ 01100000.


## Task
Create a model (an object of type `nn.Module`) that will find the value of $\phi$ for strings in the dataset. The training data consists of strings $S$ and their corresponding values $\phi(S)$. Note that the pattern set $W$ is hidden, and your task is to approximate $\phi$ without knowing it.

Your model must accept input data in the form $(\text{batch}, n)$. The output must be values in the form $(\text{batch}, 1)$ or $(\text{batch},)$, where $\text{batch}$ is the number of samples.

### Data
The data available to you in this task are:
* `train_dataset.csv` - a file with data on which you will train your model
* `val_dataset.csv` - a file with data on which you will test your model

### Evaluation Criterion
The task will be evaluated based on the [MSE](https://en.wikipedia.org/wiki/Mean_squared_error) (Mean Squared Error) metric, which is one of the most commonly used metrics for evaluating regression quality.

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
Where $y_i$ is the true value, and $\hat{y}_i$ is the value predicted by the model. The value $i$ is the sample number, and $n$ is the total number of samples.

This metric is already implemented for you in this notebook.

**Ultimately, your solution will be evaluated on a secret test set based on the MSE metric.** The test set does not differ significantly from the validation set.

- If the MSE value for your model is 64 (or more), you will receive 0 points for the task
- If the MSE value for your model is 64 (or less), you will receive X points for the task, where X is defined as follows:
$$
\text{X} = \frac{64 - MSE}{64} \times 100
$$

## Constraints
- Your solution should include an ML/DL model with learnable parameters. Purely algorithmic solutions will not be accepted.
- Your solution will be tested on the Competition Platform without internet access and in a GPU environment.
- Evaluation of your final solution on the Competition Platform cannot take longer than 4 minutes with GPU.
- Your model can be trained for a maximum of 4000 iterations, which corresponds to a single pass through the variable ```dl``` (see the example solution).
- Your model cannot have more than 50000 parameters.

## Notes and Tips
- Each string has the same fixed length.
- Each target subsequence has a length shorter than the length of the source strings.
- We consider three subsequences. Each has an assigned value, which is an integer.
- Each string may contain any number of subsequences (including none of them).
- Strings and subsequences come from a binary alphabet and are represented as lists.

## Submission Files
This notebook completed with your solution (see `YourModel` class and model training).

## Evaluation
Remember that during evaluation, the flag `FINAL_EVALUATION_MODE` will be set to `True`.

You can earn between 0 and 100 points for this task. The number of points you earn will be calculated on the (secret) test set on the Competition Platform based on the above formula, rounded to an integer. If your solution does not meet the above criteria or does not execute correctly, you will receive 0 points for the task.

# Starter Code

In this section, we initialize the environment by importing the necessary libraries and functions. The prepared code will help you operate on the data efficiently and build the proper solution.

In [1]:
######################### DO NOT CHANGE THIS CELL DURING SUBMISSION ##########################

FINAL_EVALUATION_MODE = False # During evaluation, we will set this flag to True.

In [2]:
######################### DO NOT CHANGE THIS CELL ##########################

import os
import gdown
import pandas
import torch
import numpy as np
import torch.optim as optim
import torch.nn as nn

In [3]:
######################### DO NOT CHANGE THIS CELL ##########################

def seed_everything(seed: int) -> None:
    """
    Sets the seed for reproducibility of results in Python, NumPy, and PyTorch.

    The function sets the seed for random number generators in Python, NumPy, and PyTorch,
    and configures PyTorch to operate in deterministic mode.

    Parameters:
        seed (int): The seed value to set.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
######################### DO NOT CHANGE THIS CELL ##########################

seed_everything(12345)

device = 'cuda'
assert torch.cuda.is_available(), "CUDA not available!"

## Loading Data
Using the code below, we load the data containing strings and their values.

In [5]:
######################### DO NOT CHANGE THIS CELL ##########################

class CSVDataloader(torch.utils.data.DataLoader):
    """
    The CSVDataloader class is used to load data from CSV files and return it in batches.

    Accepts:
        csv_file (str): Path to the CSV file.
        batch_size (int): Size of the batch.
        shuffle (bool): Whether to shuffle the data.
    """
    def __init__(self, csv_file, batch_size=128, shuffle=True):
        
        class CSVDataset(torch.utils.data.Dataset):
            """
            The CSVDataset class is used to store data from CSV files as individual samples.
            """
            def __init__(self, csv_file: str):
                data = pandas.read_csv(csv_file).values
                self.x = torch.tensor(data[:, :-1], dtype=torch.float32)  # Features
                self.y = torch.tensor(data[:, -1], dtype=torch.float32)  # Labels

            def __len__(self) -> int:
                return len(self.x)

            def __getitem__(self, idx: int) -> tuple:
                return self.x[idx].long(), self.y[idx]

        dataset = CSVDataset(csv_file)
        self.seq_len = dataset.x.shape[1]
        super().__init__(dataset, batch_size=batch_size, shuffle=shuffle)

In [6]:
######################### DO NOT CHANGE THIS CELL ##########################

# Initialize training dataset
train_dataset_path = "train_dataset.csv"
val_dataset_path = "val_dataset.csv"

if not os.path.exists(train_dataset_path):
    url = "https://drive.google.com/uc?id=1INeYNpPA_YwojuQbMizlsFsERJ-PJX-E"
    gdown.download(url, train_dataset_path, quiet=True)

if not os.path.exists(val_dataset_path):
    url = "https://drive.google.com/uc?id=1oQcOMyDWVX0x76LOyp4TcFip1koRuodN"
    gdown.download(url, val_dataset_path, quiet=True)

dl = CSVDataloader("train_dataset.csv")
val_dl = CSVDataloader("val_dataset.csv")

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1INeYNpPA_YwojuQbMizlsFsERJ-PJX-E

but Gdown can't. Please check connections and permissions.

## Evaluation Criterion Code

Code similar to the following will be used to evaluate the solution on the test set.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

def mse_criterium(
        estimations: torch.Tensor, 
        answers: torch.Tensor
    ) -> torch.Tensor:
    """
    Calculates the Mean Squared Error (MSE) loss between predictions and true values.

    Parameters:
        estimations (torch.Tensor): Model predictions.
        answers (torch.Tensor): True values.

    Returns:
        torch.Tensor: Mean squared error value.
    """
    return torch.mean((estimations.view(-1) - answers.view(-1)) ** 2)


def validate_model(
        model: torch.nn.Module, 
        val_dl: torch.utils.data.DataLoader,
    ) -> float:
    """
    Validates the model on the validation set. Returns the averaged
    mean squared error for all samples.

    Parameters:
        model (torch.nn.Module): Model to evaluate.
        val_dl (torch.utils.data.DataLoader): DataLoader with validation data.

    Returns:
        float: Averaged mean squared error value.
    """
    model = model.eval().to(device)
    values = []
    for x, y in val_dl:
        x = x.to(device)
        y = y.to(device)
        y_pred = model(x)

        mse = mse_criterium(y_pred, y).cpu().item()
        values.append(mse)

    final_value = torch.tensor(values).mean().item()

    return final_value

def estimate_points(mse: float) -> int:
    """
    Function determining the number of points for the task based on the mean squared error value.

    Parameters:
        mse (float): Mean squared error value.

    Returns:
        int: Number of points for the task (0 - 100).
    """
    points = max((100 * (64 - mse)) / 64, 0)
    return int(round(points))

## Example Solution
Below we present a simplified solution, which serves as an example demonstrating the basic functionality of the notebook. It can be used as a starting point for developing your solution.

A simple example could be a solution based on a multi-layered perceptron (MLP).
In this case, the binary strings are treated as input to our network, and the output of the network models the value of the given string. By minimizing the mean squared error (MSE), we teach the network to correctly estimate the value of the subsequence based on its elements.

The following illustration shows how we train our model to correctly evaluate string values.

![](https://live.staticflickr.com/65535/54328760659_2e9355bb07_c.jpg)

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

class MLP(nn.Module):
    """
    Class representing an MLP network model with four hidden layers.

    Parameters:
        input_length (int): Length of the network input (sequence length).
    """
    def __init__(self, input_length: int):
        super(MLP, self).__init__()
        neurons_num = [256, 128, 64, 32]

        self.fc_layers = nn.Sequential(
            nn.Linear(input_length, neurons_num[0]),
            nn.ReLU(),
            nn.Linear(neurons_num[0], neurons_num[1]),
            nn.ReLU(),
            nn.Linear(neurons_num[1], neurons_num[2]),
            nn.ReLU(),
            nn.Linear(neurons_num[2], neurons_num[3]),
            nn.ReLU(),
            nn.Linear(neurons_num[3], 1),
        )

        print("Number of parameters:", sum(p.numel() for p in self.parameters()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Function that takes data sequences and returns predictions of their values using the MLP network.

        Parameters:
            x (torch.Tensor): Data sequence.

        Returns:
            torch.Tensor: Predictions of sequence values.
        """
        x = x.float()
        x = self.fc_layers(x)
        return x


### Training the Example Model

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
	model = MLP(dl.seq_len).to(device)

	optimizer = optim.Adam(model.parameters(), lr=0.005)
	criterion = nn.MSELoss()

	model.train()
	for batch in iter(dl): # single iteration over dl - 4000 batches
		inputs, targets = batch
		inputs, targets = inputs.to(device).long(), targets.to(device).float().unsqueeze(1)

		optimizer.zero_grad()
		outputs = model(inputs)

		loss = criterion(outputs, targets)
		loss.backward()
		optimizer.step()


### Evaluating the Example Solution

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

# validation of the example solution
if not FINAL_EVALUATION_MODE:
    score = validate_model(model, val_dl)
    print(f"Mean Squared Error: {score:.2f}")

# Your Solution
In this section, please place your solution. Make changes only here!

In [ ]:
# example model that you can modify

class YourModel(nn.Module):
    def __init__(self, sequence_len):
        super(YourModel, self).__init__()
        self.layer = nn.Linear(sequence_len, 1)

    def forward(self, x):
        """
        Function that takes data sequences and returns predictions of their values.

        Parameters:
            x (torch.Tensor): Data sequence.

        Returns:
            torch.Tensor: Predictions of sequence values.
        """
        return self.layer(x.float())


### Training Your Model
Here, implement the training of your model.

In [ ]:
your_model = YourModel(dl.seq_len).to(device)

# ...

your_model = your_model.eval()

# Evaluation

Running the cell below will allow you to check how many points your solution would score on the validation data. Before submitting, make sure the entire notebook runs from start to finish without errors and without requiring user intervention after selecting the "Run All" option.

In [ ]:
# ######################### DO NOT CHANGE THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    assert sum(p.numel() for p in your_model.parameters()) < 50000, "Model has too many parameters"

    mse = validate_model(your_model, val_dl)
    score = estimate_points(mse)

    print(f"Mean Squared Error: {mse:.2f}")
    print(f"Estimated points for the task: {score}")

During evaluation, the model will be saved as `your_model.pkl` and evaluated on the test set.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    your_model = your_model.eval()

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(your_model, f)